# 02 — Athena External Table (Curated Parquet)

This notebook creates an **Athena external table** over the curated Parquet written by Notebook 01.

Why this exists:
- It supports the **Data Engineering** section of the Design Document.
- It gives you SQL access to curated buoy data (counts, sanity queries, etc.).

If you don't need Athena for your workflow, you can skip this notebook.


In [1]:
%pip install -q -r ../docker/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect
from pyathena.pandas.util import as_pandas

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r S3_PREFIX_PARQUET
%store -r BUOY_IDS
%store -r manifest_s3_uri

print("Bucket:", bucket)
print("Region:", region)
print("S3_PREFIX_PARQUET:", S3_PREFIX_PARQUET)
print("BUOY_IDS:", BUOY_IDS)
print("Manifest (S3):", manifest_s3_uri)

# Stronger: prefer manifest values (prevents stale %store problems)
try:
    _m = pd.read_csv(manifest_s3_uri)
    if "bucket" in _m.columns:
        bucket = _m.loc[0, "bucket"]
    if "region" in _m.columns:
        region = _m.loc[0, "region"]
    if "S3_PREFIX_PARQUET" in _m.columns:
        S3_PREFIX_PARQUET = _m.loc[0, "S3_PREFIX_PARQUET"]
    print("\n[INFO] Loaded config from manifest.")
    print("Bucket:", bucket)
    print("Region:", region)
    print("S3_PREFIX_PARQUET:", S3_PREFIX_PARQUET)
except Exception as e:
    print("[WARN] Could not load manifest; using %store values. Error:", e)

Bucket: sagemaker-us-east-1-318401170150
Region: us-east-1
S3_PREFIX_PARQUET: curated/ndbc_parquet
BUOY_IDS: ['46086', '46042', '46011']
Manifest (S3): s3://sagemaker-us-east-1-318401170150/manifests/buoy=all/curated_manifest.csv

[INFO] Loaded config from manifest.
Bucket: sagemaker-us-east-1-318401170150
Region: us-east-1
S3_PREFIX_PARQUET: curated/ndbc_parquet


In [4]:
# Athena staging dir (query results)
s3_staging_dir = f"s3://{bucket}/athena/staging/"
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)
print("Athena staging dir:", s3_staging_dir)

Athena staging dir: s3://sagemaker-us-east-1-318401170150/athena/staging/


In [5]:
database_name = "ndbc_data"
table_name = "curated_stdmet"

# Parquet root (PARTITIONED BY (buoy string) expects folders buoy=XXXX/)
s3_parquet_root = f"s3://{bucket}/{S3_PREFIX_PARQUET}/"
print("Parquet root:", s3_parquet_root)

Parquet root: s3://sagemaker-us-east-1-318401170150/curated/ndbc_parquet/


In [6]:
# Create database
with conn.cursor() as cur:
    cur.execute(f"CREATE DATABASE IF NOT EXISTS {database_name}")
print("Created database (if not exists):", database_name)

Created database (if not exists): ndbc_data


In [7]:
# Create external table (partitioned by buoy)
# IMPORTANT: If you changed the S3 location or schema, DROP the old table first.
with conn.cursor() as cur:
    cur.execute(f"DROP TABLE IF EXISTS {database_name}.{table_name}")
print("Dropped table (if existed):", f"{database_name}.{table_name}")

create_table_sql = f"""
CREATE EXTERNAL TABLE {database_name}.{table_name} (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    wind_speed_ms double,
    wave_energy double
)
PARTITIONED BY (`buoy` string)
STORED AS PARQUET
LOCATION '{s3_parquet_root}'
"""

print(create_table_sql)

with conn.cursor() as cur:
    cur.execute(create_table_sql)

print("Created table:", f"{database_name}.{table_name}")

Dropped table (if existed): ndbc_data.curated_stdmet

CREATE EXTERNAL TABLE ndbc_data.curated_stdmet (
    `timestamp` timestamp,
    station_id string,
    wind_direction double,
    wind_speed double,
    wind_gust double,
    wave_height double,
    dominant_wave_period double,
    average_wave_period double,
    mean_wave_direction double,
    pressure double,
    air_temperature double,
    water_temperature double,
    dewpoint_temperature double,
    wind_speed_ms double,
    wave_energy double
)
PARTITIONED BY (`buoy` string)
STORED AS PARQUET
LOCATION 's3://sagemaker-us-east-1-318401170150/curated/ndbc_parquet/'

Created table: ndbc_data.curated_stdmet


In [8]:
# Load partitions (buoy=XXXX folders)
repair_sql = f"MSCK REPAIR TABLE {database_name}.{table_name}"
print(repair_sql)

with conn.cursor() as cur:
    cur.execute(repair_sql)

print("Repaired partitions.")

MSCK REPAIR TABLE ndbc_data.curated_stdmet
Repaired partitions.


In [9]:
# Sanity query: row counts per buoy
q = f"""
SELECT buoy, COUNT(*) AS n
FROM {database_name}.{table_name}
GROUP BY buoy
ORDER BY n DESC
"""

cur = conn.cursor()
cur.execute(q)
df_counts = as_pandas(cur)
df_counts

,buoy,n
0,46011,53530
1,46086,51531
2,46042,12571


In [10]:
# Sample data
q = f"""SELECT * FROM {database_name}.{table_name} LIMIT 10"""

cur = conn.cursor()
cur.execute(q)
as_pandas(cur)

,timestamp,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,pressure,air_temperature,water_temperature,dewpoint_temperature,wind_speed_ms,wave_energy,buoy
0,2023-01-01 01:40:00,46042,333.0,14.8,19.1,4.17,14.81,7.12,304.0,None,None,None,None,None,None,46042
1,2023-01-01 03:40:00,46042,320.0,12.0,16.7,3.87,12.90,7.47,288.0,None,None,None,None,None,None,46042
2,2023-01-01 04:40:00,46042,313.0,12.0,16.0,3.83,14.81,7.37,296.0,None,None,None,None,None,None,46042
3,2023-01-01 09:40:00,46042,329.0,11.6,14.2,4.38,9.09,7.72,300.0,None,None,None,None,None,None,46042
4,2023-01-01 10:40:00,46042,323.0,13.2,17.5,4.78,10.00,7.86,306.0,None,None,None,None,None,None,46042
5,2023-01-01 11:40:00,46042,323.0,13.5,16.8,5.10,10.81,8.22,309.0,None,None,None,None,None,None,46042
6,2023-01-01 05:40:00,46042,317.0,11.6,15.1,3.83,12.12,7.37,288.0,None,None,None,None,None,None,46042
7,2023-01-01 06:40:00,46042,317.0,12.8,16.5,3.92,7.14,7.21,294.0,None,None,None,None,None,None,46042
8,2023-01-01 07:40:00,46042,319.0,13.3,15.9,3.96,13.79,7.13,286.0,None,None,None,None,None,None,46042
9,2023-01-01 08:40:00,46042,326.0,12.4,15.9,4.07,8.33,7.49,303.0,None,None,None,None,None,None,46042


In [11]:
%store database_name
%store table_name

Stored 'database_name' (str)
Stored 'table_name' (str)
